# Lab 5: Fine-tuning a coding agent with SFT

## Notebook 2: Fine-tune with a serverless LoRA job

We adapt the 4B base model to Python code generation using **LoRA** on SageMaker AI
serverless customization: no cluster to provision, no container to build. You never choose
a training instance; SageMaker provisions the compute and bills per token processed.

### Choosing a base model

The base model is configured in [`config.py`](config.py). To browse customizable JumpStart
models, run the cell below. To switch, update `BASE_MODEL_ID` in `config.py`, all notebooks
pick up the change.

> **Note:** Not every JumpStart model supports every technique. A "No recipes found" error
> means the model does not support SFT. Serving is a separate question, check the model can
> be served by the LMI/DJL container (notebook 4) before committing to it.

In [ ]:
import boto3

from config import BASE_MODEL_ID

sm = boto3.client("sagemaker")
models, kwargs = [], {"HubName": "SageMakerPublicHub", "HubContentType": "Model",
                      "MaxResults": 100}
while True:
    r = sm.list_hub_contents(**kwargs)
    for item in r["HubContentSummaries"]:
        if "@capability:customization" in item.get("HubContentSearchKeywords", []):
            models.append(item["HubContentName"])
    if "NextToken" in r:
        kwargs["NextToken"] = r["NextToken"]
    else:
        break

print(f"base model for this lab: {BASE_MODEL_ID}\n")
print(f"customizable models available ({len(models)}):")
print("\n".join(sorted(models)))

In [ ]:
%load_ext autoreload
%autoreload 2

#### Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from config import BASE_MODEL_ID, DATASET_PREFIX

base_model_id = BASE_MODEL_ID
training_dataset = DataSet.get(name=f"{DATASET_PREFIX}-train")
val_dataset = DataSet.get(name=f"{DATASET_PREFIX}-val")

output_path = (f"s3://{bucket_name}/{default_prefix}/{base_model_id}-coding-agent"
               if default_prefix else f"s3://{bucket_name}/{base_model_id}-coding-agent")
print(f"training output: {output_path}")

### Create the Model Package Group

The trained model is registered here, and notebooks 3 and 4 look it up by this name.

In [ ]:
import hashlib

from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

# SageMaker caps Model Package Group names at 63 characters. Hash-truncate when
# base_model_id + suffix exceeds it, so every notebook derives the same name.
MAX_MPG_NAME_LENGTH = 63
suffix = "-coding-agent-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

print(f"Model Package Group: {model_package_group_name}")

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

try:
    ModelPackageGroup.get(model_package_group_name=model_package_group_name)
    print(f"already exists: {model_package_group_name}")
except ClientError:
    ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Coding-agent Python generation, serverless SFT",
    )
    print(f"created: {model_package_group_name}")

### Configure the trainer

In [ ]:
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

MAX_JOB_NAME_LENGTH, TIMESTAMP_LENGTH = 63, 15
base_job_name = "coding-agent-sft"[: MAX_JOB_NAME_LENGTH - TIMESTAMP_LENGTH].rstrip("-")

trainer = SFTTrainer(
    model=base_model_id,
    training_type=TrainingType.LORA,
    model_package_group=model_package_group_name,
    training_dataset=training_dataset,
    validation_dataset=val_dataset,
    s3_output_path=output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True,
    base_job_name=base_job_name,
)

print(f"fine-tuning:    {base_model_id}")
print(f"training type:  {TrainingType.LORA.value}")
print(f"model package:  {model_package_group_name}")
print(f"training data:  {training_dataset.name} v{training_dataset.version}")
print(f"validation:     {val_dataset.name} v{val_dataset.version}\n")

print("default hyperparameters:")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

### Hyperparameters

The values below match this lab's training recommendation: LoRA rank 16 (alpha 32), one
epoch over the 10K subset, a 1,024-token sequence cap, and a 2e-4 learning rate.

> **Confirm the recipe's names first.** The serverless recipe abstracts some knobs, and not
> every field is guaranteed to be exposed by every model's recipe. The cell above printed
> the recipe defaults; set only fields that appear there. A field the recipe does not accept
> raises rather than being silently ignored, so if a line errors, comment it out and rely on
> the recipe default. `global_batch_size` is the effective batch (micro-batch and gradient
> accumulation are chosen for you).

Code solutions in this dataset are short, so `dataset_max_len = 1024` keeps essentially all
of them intact. Print the recipe default before overriding it.

In [ ]:
print(f"recipe default dataset_max_len: {trainer.hyperparameters.dataset_max_len}")

trainer.hyperparameters.learning_rate = 0.0002
trainer.hyperparameters.global_batch_size = 16
trainer.hyperparameters.max_epochs = 1
trainer.hyperparameters.lr_warmup_steps_ratio = 0.1
trainer.hyperparameters.lora_rank = 16
trainer.hyperparameters.lora_alpha = 32
trainer.hyperparameters.dataset_max_len = 1024

# Some recipes expose the LR schedule under this name; set it if present, else skip.
try:
    trainer.hyperparameters.lr_scheduler = "cosine"
except Exception as e:
    print(f"lr_scheduler not settable on this recipe ({type(e).__name__}); using default")

records = len(training_dataset) if hasattr(training_dataset, "__len__") else 9800
gbs = int(trainer.hyperparameters.global_batch_size)
epochs = int(trainer.hyperparameters.max_epochs)
steps = records * epochs // gbs
print(f"\n~{records} records x {epochs} epoch / batch {gbs} = ~{steps} optimizer steps\n")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

### Launch

`wait=False` returns immediately. Target window is roughly **15-20 minutes** for the single
epoch. A fixed part of that is SageMaker provisioning compute rather than training, so the
wall-clock does not scale linearly with epochs.

In [ ]:
training_job = trainer.train(wait=False)
TRAINING_JOB_NAME = training_job.training_job_name
print(f"launched: {TRAINING_JOB_NAME}")

Poll until the job reaches `Completed`, then continue to **notebook 3**.

In [ ]:
import time

from sagemaker.core.resources import TrainingJob

while True:
    j = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
    print(f"{j.training_job_status} / {j.secondary_status}")
    if j.training_job_status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(120)